# 🕵️ Dia 61 — Triagem estática de um arquivo suspeito

**Investigador Cyber 90 Dias** · Fase 4 · Rank Jutsu `B`

> Baseado no *Day 61: Static triage of a suspicious file* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-4-digital-forensics-incident-investigation/day-61.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

A TI da Serra Azul mandou a pasta `AppData\Roaming\MSA` do `SA-CX-07` e perguntou: *"Vocês podem rodar para ver o que faz?"* Rafa: *"Ninguém roda nada. E ninguém sobe isso no VirusTotal. Hash primeiro."*

Neste notebook você vai:

1. **Identificar o tipo pelos *magic bytes***, não pela extensão.
2. **Calcular SHA-256, SHA-1 e MD5** num único passe.
3. **Reimplementar o `strings`** (ASCII e UTF-16LE) e separar indicadores de contexto.
4. **Consultar o CIRCL hashlookup só com o hash** (plano B offline) e conferir o hash do EICAR **sem criar o arquivo**.
5. **Medir a entropia** e ver por que arquivos compactados parecem aleatórios.
6. **Gerar `cards.md`** e jogar o quiz "o que vai no cartão?".

> ⚠️ **Nenhum malware aqui.** Os únicos arquivos são: o `config.ini` sintético do caso (texto puro), arquivos de teste que o próprio notebook cria só com cabeçalhos (não executam nada) e, se existir na máquina, um binário legítimo do sistema (`/usr/bin/curl` ou o próprio Python). A *string* EICAR **não** aparece neste notebook: só o hash dela.

## 1. Carregando o `config.ini` do caso

In [ ]:
# ------------------------------------------------------------
# CARREGADOR OFFLINE-FIRST
# Tenta baixar cada arquivo do GitHub (raw). Se a rede falhar,
# usa a cópia EMBUTIDA aqui embaixo (comprimida para caber no notebook).
# Nos dois casos, calculamos o SHA-256: é a nossa "cadeia de custódia" de bolso.
# ------------------------------------------------------------
import base64          # para decodificar as cópias embutidas
import hashlib         # SHA-256 de cada arquivo carregado
import urllib.request  # download simples, sem bibliotecas externas
import zlib            # as cópias embutidas estão comprimidas com zlib

# Endereço "cru" dos dados do LAB-P4 no repositório original (ThiefDroppers)
RAW_LAB = ("https://raw.githubusercontent.com/thiefdroppers/cyber-investigator-skills/main/"
           "curriculum/phase-4-digital-forensics-incident-investigation/resources/case-lab-p4/")
# Endereço "cru" dos dados sintéticos da Operação Isca neste repositório
RAW_BR = ("https://raw.githubusercontent.com/Ridd1kulusC0d3r/investigador-cyber-90-dias/main/"
          "caso-continuo/dados/")

# Plano B: cópias fiéis dos arquivos, feitas a partir do repositório local.
EMBUTIDOS = {
    "br:dia-61/config.ini": """
eNpVkk1v2kAQhu+W/B/mgpJIwR9JgCbIBysQyVLBFJz2UCprYk/ISrted3dNKL++Y1Sa9ra78/W+z84U0vWX5+xrDptsWcyL7DGH
2RyK9Txbpov5sshhCHlLBivUkNkK4TJr9mSd2GGtDTz+eiED9xHMBNor35vCkhMJFMp3NHQNTX89UNU55HONAcy0Eo3QFgI6oGol
weX66RFuxtH46hqylYX+OprcToK+35pao+vuCBp+dgSdgko3r2IHNcEFug6lOPZSLuAVpdUcta5T2Gc5MtfQokFwRuCOFLBwdKJC
7vx9x67kD99rtKJkwSOk7ntuaNcZbNjphgyXpsdO+t6ejEWd3ARxMPE90XAfyVNLUskgXa1maZEOtotNulUWh/+oYo/ke45RvGLC
cUj/xHqeaee0OulhOayXeWgWZMnsBdcmb8619iEMz/2G6qTyzC3EVoT7m7AVze6jqjTUH/FvdXz/KRjFQRxFQXwX/V8kekZ7lLq0
bLuptU1uo4jtIgNjiR/PMb9Ofa/j3iWjbBwjOwopMRwFEVx+E5z1bmFZQBwF0RT4YXw3hcP47grYd8jkTiYlOWSP1OwFmrJFBpkM
njfz9WqdP2Wf54PtTFed4gHW96RQwlGpXpLbfhWKfJZDS7yF7POBP1VXaOBsHJDNWKgReGEOol8BxoTNG9P9DceT9Uw=""",
}

REGISTRO_DE_CARGA = []   # (arquivo, origem, sha256): vai para as suas notas de método


def carregar(chave):
    """chave = 'lab:logs/firewall.log' ou 'br:dia-57/proxy-sa.log'. Devolve o texto."""
    prefixo, rel = chave.split(":", 1)                    # separa de onde vem e o caminho
    url = (RAW_LAB if prefixo == "lab" else RAW_BR) + rel  # monta a URL raw
    reserva = zlib.decompress(base64.b64decode(EMBUTIDOS[chave].replace("\n", "")))  # bytes da cópia embutida
    try:
        with urllib.request.urlopen(url, timeout=10) as r:  # tenta a rede, com tempo limite
            dados, origem = r.read(), "rede"
    except Exception:                                       # sem rede, 404, timeout: tanto faz
        dados, origem = reserva, "embutido"
    if origem == "rede" and dados != reserva:              # a versão online mudou desde que copiamos?
        print(f"⚠️  {rel}: versão online difere da embutida; usando a embutida (números do dia batem com ela)")
        dados, origem = reserva, "embutido (online divergente)"
    sha = hashlib.sha256(dados).hexdigest()                # impressão digital do que vamos analisar
    REGISTRO_DE_CARGA.append((rel, origem, sha))           # anota para o relatório
    return dados.decode("utf-8")


print(f"{len(EMBUTIDOS)} arquivos disponíveis. Use carregar('...').")

In [ ]:
from pathlib import Path
Path("triagem").mkdir(exist_ok=True)                              # nossa "área de triagem"
CONFIG = Path("triagem/config.ini")
CONFIG.write_bytes(carregar("br:dia-61/config.ini").encode("utf-8"))  # bytes exatamente como na fonte
print(CONFIG, CONFIG.stat().st_size, "bytes")

In [ ]:
# Tabela de carga: o que foi lido, de onde e com qual hash.
for rel, origem, sha in REGISTRO_DE_CARGA:     # uma linha por arquivo
    print(f"{rel:40s} {origem:10s} {sha[:16]}…")  # 16 primeiros caracteres bastam para conferir de olho

## 2. Tipo pelos *magic bytes*

A tabela do Conceito em código. Criamos quatro arquivos de teste **com a extensão errada de propósito**: só o cabeçalho, sem nada executável.

In [ ]:
MAGICOS = [                                    # (assinatura no início do arquivo, descrição)
    (b"MZ", "executável Windows (PE/DOS)"),
    (b"PK\x03\x04", "zip (também .docx, .xlsx, .jar, .apk)"),
    (b"%PDF", "PDF"),
    (b"\x7fELF", "binário Linux (ELF)"),
    (b"\xd4\xc3\xb2\xa1", "captura pcap (little-endian)"),
]

def tipo_por_magic(caminho):
    cabeca = Path(caminho).read_bytes()[:8]                   # só os primeiros bytes importam
    for assinatura, descricao in MAGICOS:
        if cabeca.startswith(assinatura):
            return descricao
    try:
        cabeca.decode("ascii")                                 # tudo imprimível? provavelmente texto
        return "texto (sem assinatura binária conhecida)"
    except UnicodeDecodeError:
        return "desconhecido"

# Arquivos de teste: cabeçalho + zeros. NÃO são programas: não há código nenhum depois do cabeçalho.
TESTES = {"fatura.pdf": b"MZ" + b"\x00" * 62,                    # "PDF" que é executável
          "foto.jpg": b"PK\x03\x04" + b"\x00" * 26,              # "foto" que é zip
          "leia-me.txt": b"%PDF-1.7\n%%EOF\n",                    # "texto" que é PDF
          "relatorio.docx": b"\x7fELF" + b"\x00" * 60}            # "docx" que é ELF
for nome, conteudo in TESTES.items():
    (Path("triagem") / nome).write_bytes(conteudo)
for nome in list(TESTES) + ["config.ini"]:
    print(f"{nome:16s} → {tipo_por_magic(Path('triagem') / nome)}")

## 3. Os três hashes, num passe só

In [ ]:
import hashlib

def hashes(caminho, bloco=1 << 20):
    """Lê o arquivo em blocos de 1 MiB (funciona com arquivos grandes) e alimenta os três algoritmos."""
    h = {"sha256": hashlib.sha256(), "sha1": hashlib.sha1(), "md5": hashlib.md5()}
    with open(caminho, "rb") as f:
        for pedaco in iter(lambda: f.read(bloco), b""):        # até o fim do arquivo
            for alg in h.values():
                alg.update(pedaco)
    return {nome: alg.hexdigest() for nome, alg in h.items()}

H_CONFIG = hashes(CONFIG)
for k, v in H_CONFIG.items():
    print(f"{k:7s} {v}")
ESPERADO = "c1581726b27e0d672fa76856638b6ed3dcd5e1ae4b638fbd3a106cafab22d039"
print("bate com o Checkpoint BR?", "✅" if H_CONFIG["sha256"] == ESPERADO else "❌ (o arquivo mudou? registre!)")

In [ ]:
# A amostra "conhecida como boa" do original: um binário legítimo que já está na máquina.
import shutil, sys
candidatos = [shutil.which("curl"), "/usr/bin/curl", sys.executable]    # curl, ou o próprio Python
SAMPLE01 = next((Path(c) for c in candidatos if c and Path(c).is_file()), None)
if SAMPLE01:
    copia = Path("triagem/sample01")
    copia.write_bytes(SAMPLE01.read_bytes())                    # trabalhamos sempre numa CÓPIA
    H_SAMPLE = hashes(copia)
    print(f"sample01 = cópia de {SAMPLE01} ({copia.stat().st_size} bytes)")
    print("tipo:", tipo_por_magic(copia))
    print("sha256:", H_SAMPLE["sha256"])
else:
    H_SAMPLE = None
    print("Nenhum binário legítimo encontrado aqui; pule para o config.ini.")

## 4. Reputação: só o hash sai da máquina

CIRCL hashlookup: `200` com JSON quando conhece o arquivo; `404` ("Non existing") quando não conhece. Sem rede, o notebook diz isso e segue.

In [ ]:
import json, urllib.request, urllib.error
from datetime import datetime, timezone

def circl(sha256):
    url = f"https://hashlookup.circl.lu/lookup/sha256/{sha256}"
    quando = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")     # reputação sempre DATADA
    try:
        with urllib.request.urlopen(url, timeout=10) as r:
            dados = json.load(r)
            return f"CIRCL {quando}: conhecido — {dados.get('FileName', '?')} ({dados.get('ProductName', 'sem produto')})"
    except urllib.error.HTTPError as e:
        if e.code == 404:
            return f"CIRCL {quando}: Non existing (sem registro: limpo, novo ou direcionado)"
        return f"CIRCL {quando}: erro HTTP {e.code}"
    except Exception:
        return f"CIRCL {quando}: não consultado (sem rede). Consulte depois e registre."

REP_CONFIG = circl(H_CONFIG["sha256"])
REP_SAMPLE = circl(H_SAMPLE["sha256"]) if H_SAMPLE else "—"
print("config.ini:", REP_CONFIG)
print("sample01  :", REP_SAMPLE)

In [ ]:
# EICAR: o original manda consultar SÓ o hash. Não criamos o arquivo (o antivírus da máquina reagiria).
EICAR_SHA256 = "275a021bbfb6489e54d471899f7db9d1663fc695ec2fe2a2c4538aabf651fd0f"
print("Hash EICAR (68 bytes):", EICAR_SHA256)
print("Busque-o no VirusTotal (só a caixa de busca) e anote: nº de motores e 3 nomes de detecção.")
print(circl(EICAR_SHA256))

## 5. `strings` em Python: ASCII e UTF-16LE

In [ ]:
import re

def strings(dados, minimo=8):
    """Sequências imprimíveis de pelo menos 'minimo' caracteres, nas duas codificações."""
    ascii_ = [m.group().decode() for m in re.finditer(rb"[\x20-\x7e]{%d,}" % minimo, dados)]
    utf16 = [m.group().decode("utf-16le") for m in re.finditer(rb"(?:[\x20-\x7e]\x00){%d,}" % minimo, dados)]
    return ascii_, utf16

INDICADORES = {                                                 # o que vira IOC no cartão
    "url": re.compile(r"https?://[^\s\"']+"),
    "ip": re.compile(r"\b(?:\d{1,3}\.){3}\d{1,3}\b"),
    "caminho_windows": re.compile(r"%\w+%\\[^\s]+|[A-Za-z]:\\[^\s]+"),
    "user_agent": re.compile(r"Mozilla/\d\.\d.*"),
    "tarefa": re.compile(r"^tarefa=(.*)$"),
}

def classificar(lista):
    achados = []
    for s in lista:
        for tipo, rx in INDICADORES.items():
            for m in rx.finditer(s):
                achados.append((tipo, m.group(1) if m.groups() else m.group()))
    return sorted(set(achados))

a, u = strings(CONFIG.read_bytes())
print(f"config.ini: {len(a)} strings ASCII, {len(u)} UTF-16LE")
for tipo, valor in classificar(a):
    print(f"  🔎 {tipo:16s} {valor}")

In [ ]:
# O mesmo no binário legítimo: muitas strings "com cara de rede" e nada de suspeito por isso.
if H_SAMPLE:
    a, u = strings(Path("triagem/sample01").read_bytes())
    urls = sorted({v for t, v in classificar(a) if t == "url"})
    print(f"sample01: {len(a)} ASCII, {len(u)} UTF-16LE, {len(urls)} URLs distintas. Exemplos:")
    for x in urls[:5]:
        print("  ", x)
    print("➡️  Um programa de rede legítimo tem strings de rede. Strings sugerem capacidade, não intenção.")

## 6. Entropia: por que arquivos compactados "parecem aleatórios"

Entropia de Shannon por byte: 0 (tudo igual) a 8 (aleatório). Texto fica entre 4 e 5; dados compactados ou cifrados, perto de 8. É a métrica por trás do "seções de alta entropia" do Conceito.

In [ ]:
import math, os, zlib
from collections import Counter

def entropia(dados):
    if not dados:
        return 0.0
    cont = Counter(dados)                                             # frequência de cada byte
    n = len(dados)
    return abs(sum(c / n * math.log2(c / n) for c in cont.values()))   # abs() evita o '-0.00' quando tudo é igual

amostras = {"config.ini (texto)": CONFIG.read_bytes(),
            "zeros": bytes(4096),
            "config.ini comprimido (zlib)": zlib.compress(CONFIG.read_bytes() * 50, 9),
            "aleatório (os.urandom)": os.urandom(4096)}
for nome, dados in amostras.items():
    e = entropia(dados)
    print(f"{nome:30s} {e:4.2f} {'█' * int(e * 4)}")

## 7. Os cartões (`cards.md`)

In [ ]:
#@title Resultados que você anotou no VirusTotal { display-mode: "form" }
data_consulta_vt = "AAAA-MM-DD"  #@param {type:"string"}
motores_eicar = "?/??"  #@param {type:"string"}
nomes_eicar = "EICAR-Test-File, EICAR_Test_File, (terceiro nome)"  #@param {type:"string"}

def cartao(titulo, campos):
    linhas = [f"## {titulo}", "", "| Campo | Registro |", "|---|---|"]
    linhas += [f"| {k} | {v} |" for k, v in campos]
    return "\n".join(linhas) + "\n"

CARTOES = []
if H_SAMPLE:
    CARTOES.append(cartao("sample01 (binário legítimo)", [
        ("Caso / item", "Treino / sample01"), ("Nome e caminho", f"cópia de {SAMPLE01}"),
        ("Tipo (magic)", tipo_por_magic("triagem/sample01")),
        ("Hashes", f"SHA-256 {H_SAMPLE['sha256']}<br>SHA-1 {H_SAMPLE['sha1']}<br>MD5 {H_SAMPLE['md5']}"),
        ("Reputação", f"{REP_SAMPLE}; VirusTotal {data_consulta_vt}: (anote)"),
        ("Explicação", "As strings de rede são esperadas numa ferramenta de rede legítima; strings sugerem capacidade, não intenção."),
        ("Decisão de triagem", "Nenhuma ação adicional")]))
CARTOES.append(cartao("Hash do teste EICAR", [
    ("Caso / item", "Treino / EICAR (só hash; arquivo não criado)"), ("Tipo (magic)", "não disponível: arquivo não criado de propósito"),
    ("Hashes", f"SHA-256 {EICAR_SHA256}"),
    ("Reputação", f"VirusTotal {data_consulta_vt}: {motores_eicar}; nomes: {nomes_eicar}"),
    ("Explicação", "Cada fornecedor dá o próprio rótulo; nomes de detecção não são taxonomia compartilhada."),
    ("Decisão de triagem", "Nenhuma (arquivo de teste)")]))
CARTOES.append(cartao("synchelper.exe (LAB-P4, sem bytes)", [
    ("Caso / item", "LAB-P4 / EVID-002 (MFT), EVID-003 (memória)"),
    ("Nome e caminho", "C:\\Users\\acct.clerk01\\AppData\\Roaming\\SyncHelper\\synchelper.exe"),
    ("Tamanho", "412.160 bytes (entrada 88413 da MFT)"),
    ("Primeira observação", "$FN criado 2026-03-13 16:42:30 UTC; processo 16:42:31 UTC"),
    ("Hashes", "Não disponíveis. Solicitar: arquivo da imagem de disco ou dump do processo (windows.pslist --pid 7488 --dump)"),
    ("Reputação", "Não consultada (sem hash)"),
    ("Relacionados", "Pai powershell.exe 7316; filho rundll32.exe 7704 com região MZ injetada; conexões para 198.51.100.23:443"),
    ("Decisão de triagem", "Adquirir e calcular hash; triagem estática em VM isolada; sandbox só privada")]))
iocs = "; ".join(f"{t}: {v}" for t, v in classificar(strings(CONFIG.read_bytes())[0]))
CARTOES.append(cartao("➕ config.ini (Isca, bytes disponíveis)", [
    ("Caso / item", "Isca / SA-CX-07, pasta AppData\\Roaming\\MSA"),
    ("Nome e caminho", "C:\\Users\\cx.centro07\\AppData\\Roaming\\MSA\\config.ini"),
    ("Tamanho", f"{CONFIG.stat().st_size} bytes (confere com a MFT 120946: 744)"),
    ("Tipo (magic)", tipo_por_magic(CONFIG)),
    ("Hashes", f"SHA-256 {H_CONFIG['sha256']}<br>SHA-1 {H_CONFIG['sha1']}<br>MD5 {H_CONFIG['md5']}"),
    ("Reputação", REP_CONFIG),
    ("Indicadores (strings)", iocs),
    ("Relacionados", "intervalo_segundos=300 é CONSISTENTE com o ritmo medido no Dia 59 (não confirma que este arquivo o controla)"),
    ("Decisão de triagem", "Arquivo de configuração em texto; IOCs vão para o Dia 60/63; executável irmão segue para sandbox privada")]))
CARTOES.append(cartao("➕ msa-atualizador.exe (Isca, sem bytes)", [
    ("Caso / item", "Isca / SA-CX-07"), ("Nome e caminho", "C:\\Users\\cx.centro07\\AppData\\Roaming\\MSA\\msa-atualizador.exe"),
    ("Tamanho", "389.120 bytes (MFT 120945)"),
    ("Primeira observação", "$FN criado 2026-10-05 11:24:29 UTC ($SI diz 2021-02-03: divergência, ver Dia 52); processo 11:24:31Z (4688)"),
    ("Hashes", "Não disponíveis. Solicitar: cópia da TI por canal com custódia (hash na origem)"),
    ("Reputação", "Não consultada (sem hash)"),
    ("Relacionados", "Pai explorer.exe; filho schtasks.exe 11:24:44Z; conexões a atualiza-modulo.example a cada ~300 s (proxy/firewall)"),
    ("Decisão de triagem", "Só sandbox privada: arquivo de estação de cooperativa (sigilo bancário, LGPD) e possivelmente direcionado")]))
Path("cards.md").write_text("# Cartões de triagem\n\n" + "\n".join(CARTOES), encoding="utf-8")
print(Path("cards.md").read_text(encoding="utf-8")[:1500], "…")

## 8. 🎮 Quiz: o que vai no cartão?

Checkpoint BR: pelo menos 5 de 6.

In [ ]:
# ------------------------------------------------------------
# QUIZ: função genérica de múltipla escolha
# MODO_JOGO = True  -> pergunta e conta acertos
# MODO_JOGO = False -> só mostra gabarito e explicação
# ------------------------------------------------------------
def jogar_quiz(perguntas, modo_jogo=True, minimo=None):
    acertos = 0                                            # placar
    for n, (enunciado, opcoes, certa, porque) in enumerate(perguntas, start=1):
        print(f"\n[{n}/{len(perguntas)}] {enunciado}")    # o enunciado
        for letra, texto in opcoes.items():                # as alternativas a), b), c)
            print(f"   {letra}) {texto}")
        if modo_jogo:
            resposta = ""
            for _ in range(5):                             # até 5 tentativas de digitar uma letra válida
                resposta = input("   Sua resposta: ").strip().lower()
                if resposta in opcoes:
                    break
            ok = (resposta == certa)                       # acertou?
            acertos += ok
            print("   ✅ Acertou!" if ok else f"   ❌ Era {certa}).")
        else:
            print(f"   Gabarito: {certa})")
        print(f"   Por quê: {porque}")
    if modo_jogo:
        print(f"\n🏁 Resultado: {acertos}/{len(perguntas)}")
        if minimo is not None:
            print("Checkpoint BR cumprido. 🥷" if acertos >= minimo
                  else f"Checkpoint BR pede pelo menos {minimo}. Releia o README e tente de novo.")
    return acertos

In [ ]:
MODO_JOGO = True
PERGUNTAS = [
    ("O arquivo se chama fatura.pdf e começa com 'MZ'. No campo Tipo vai:",
     {"a": "PDF", "b": "executável Windows (magic MZ), com extensão .pdf", "c": "desconhecido"},
     "b", "O tipo vem dos bytes; a extensão enganosa é um achado à parte."),
    ("Um motor chama o arquivo de 'Trojan.Banker.X'; os outros 69 não detectam. No cartão:",
     {"a": "Família: Trojan.Banker", "b": "Reputação (data): 1/70, rótulo de um fornecedor", "c": "Malicioso confirmado"},
     "b", "Nome de família de um só motor não é base para achado."),
    ("CIRCL respondeu 'Non existing'. Isso significa:",
     {"a": "malicioso", "b": "sem registro: limpo, novo ou direcionado", "c": "arquivo corrompido"},
     "b", "Ausência de registro é ausência de opinião."),
    ("`intervalo_segundos=300` no config.ini e o beacon de ~300 s do Dia 59. No cartão:",
     {"a": "confirma que o config controla o beacon", "b": "é consistente com o ritmo observado", "c": "não tem relação"},
     "b", "Consistente, não confirmado: falta ver o executável lendo esse arquivo."),
    ("O técnico pergunta se pode subir o msa-atualizador.exe no VirusTotal 'para agilizar':",
     {"a": "pode, é só um executável", "b": "não: consulte o hash; o arquivo pode ter dado do cliente e alertar o atacante",
      "c": "pode, se for pela VPN"},
     "b", "Busque o hash, não o arquivo."),
    ("Entropia de 7,9 bits/byte numa seção sugere:",
     {"a": "texto em português", "b": "dados compactados ou cifrados (possível packer)", "c": "arquivo vazio"},
     "b", "Perto de 8 = aparência aleatória. Sugere; não prova."),
]
jogar_quiz(PERGUNTAS, MODO_JOGO, minimo=5)

## 9. (Opcional) As ferramentas do original

In [ ]:
!file triagem/*
!strings -n 8 -e l triagem/sample01 | head
!pip -q install pefile
print("pefile é opcional: lê seções e importações de executáveis PE (use só em VM de análise).")

In [ ]:
try:
    from google.colab import files
    files.download("cards.md")
except ImportError:
    print("(Fora do Colab: cards.md ficou na pasta atual.)")

## ✅ Antes de fechar o Dia 61

- [ ] `cards.md` com os três cartões do original e os dois da Isca
- [ ] Toda reputação com data
- [ ] Nenhuma afirmação sobre o que o `synchelper.exe` ou o `msa-atualizador.exe` fazem
- [ ] Quiz com pelo menos 5/6

**Amanhã (Dia 62):** *sandbox*. Você vai ler um relatório de um instalador legítimo e aprender a separar o que a *sandbox* viu do que ela pontuou.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>